# 07 — Calibration, precision and classification of the production posterior (C2, C3, C4, C5)
On the L-sets (1000 fresh windows per scenario, production conditions) and a fresh SBC set: SBC (interior),
TARP, coverage, bias, posterior sd vs the Bayesian CRB (notebook 04), four-class and three-way classification.

In [ ]:
import json, sys, time
from dataclasses import replace
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
NB_DIR = Path.cwd().resolve()
assert NB_DIR.name == "notebooks" and NB_DIR.parent.name == "system_III", f"run from system_III/notebooks (cwd {NB_DIR})"
ROOT = NB_DIR.parent
sys.path.insert(0, str(ROOT / "src"))
import importlib, system_iii
from system_iii import physics as ph, simulator as sim, features as fe, data as da, scenarios as scn
from system_iii import sensors as sen, information as info, config as cfg, rules, metrics as wm
for _m in (ph, sim, fe, da, scn, sen, info, cfg, rules, wm):
    importlib.reload(_m)
R, FIGS, DATA = ROOT / "results", ROOT / "figures", ROOT / "data"
for _d in (R, FIGS, DATA):
    _d.mkdir(exist_ok=True)
P = ph.PLANT
T0 = time.time()

In [ ]:
from system_iii import inference as inf
import pickle
PROD = json.loads((R / "06_production.json").read_text())
_m = [pickle.load(open(f, "rb")) for f in PROD["member_files"]]
post = inf.EnsemblePosterior(_m) if PROD["use_ensemble"] else _m[0]
print("production:", PROD["production_budget"], "ensemble" if PROD["use_ensemble"] else "single", PROD["feature_set"])

In [ ]:
L = da.load_npz(DATA / "L_prod.npz")
sbc = da.make_windows(sim.prior_draws(2000, 7100), 7100, feature_set=PROD["feature_set"])
smp_sbc, leak = inf.sample_within_prior(post, sbc["x"], 1000, rng_seed=1)
cal = wm.evaluate_sbc_set(smp_sbc, sbc["theta"], inf.LO, inf.HI, seed=0); cal["leak_mean"] = float(leak.mean())
display(pd.Series(cal).round(4))
smp, leakL = inf.sample_within_prior(post, L["x"], 1000, rng_seed=2)
overall, sc_rows = wm.evaluate_scenario_set(smp, L["theta"], L["sc_name"], inf.LO, inf.HI)
tab = pd.DataFrame(sc_rows).set_index("scenario"); display(tab.round(4)); print(overall)
tw_a, tw_b = inf.three_way(smp)
three = pd.DataFrame({k: {"alpha_fault": (tw_a[L["sc_name"] == k] == 1).mean(), "alpha_undet": (tw_a[L["sc_name"] == k] == 0).mean(),
                          "beta_fault": (tw_b[L["sc_name"] == k] == 1).mean(), "beta_undet": (tw_b[L["sc_name"] == k] == 0).mean()}
                      for k in scn.CLOSED}).T
display(three.round(3))
crb = pd.read_csv(R / "04_crb.csv"); crb = crb[crb.nuisances == "all_offsets"].set_index("scenario")
tab["sd_b_over_crb"] = [tab.loc[k, "sd_b"] / crb.loc[k, "sd_beta"] if k in crb.index else np.nan for k in tab.index]
tab["sd_a_over_crb"] = [tab.loc[k, "sd_a"] / crb.loc[k, "sd_alpha"] if k in crb.index else np.nan for k in tab.index]
tab.to_csv(R / "07_scenarios.csv"); three.to_csv(R / "07_three_way.csv"); da.save_json(R / "07_calibration.json", dict(sbc=cal, overall=overall))

## Interpretation — to be written after the run
Write toward the storyline (`docs/system_III_plan.md` §1) and update the claims ledger (§2).